
# Uplift Modelling: Who Should We Actually Contact?

The churn model ranks customers by how likely they are to leave. A retention
campaign then contacts the top of that list. This notebook shows that step is
close to worthless, and why.

The campaign should reach customers whose behaviour the contact *changes*. Four
groups exist:

| | Contacted | Not contacted |
| --- | --- | --- |
| **Sure thing** | stays | stays |
| **Persuadable** | stays | leaves |
| **Lost cause** | leaves | leaves |
| **Sleeping dog** | leaves | stays |

Only persuadables repay the spend. Sure things and lost causes waste it.
Sleeping dogs are actively harmed by being contacted. A churn model cannot
distinguish any of them -- it ranks by outcome, not by responsiveness.

Separating them requires randomised data, because responsiveness is a causal
quantity: we need comparable customers observed under both arms.
`data/campaign.csv` is that experiment.


## 1. Setup

In [1]:

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.evaluation import bootstrap_metric, format_ci, paired_bootstrap
from src.features import PREDICTION_DATE, build_features
from src.generate_data import PREDICTION_DAY, START_DATE, expected_orders, generate
from src.scoring import (
    CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, NUMERIC_FEATURES,
    add_rate_features,
)

RANDOM_STATE = 42

def make_pipeline(numeric=None, categorical=CATEGORICAL_FEATURES, derive=True):
    steps = []
    if derive:
        steps.append(("rates", FunctionTransformer(add_rate_features)))
    transformers = [("num", Pipeline([
        ("i", SimpleImputer(strategy="median")), ("s", StandardScaler()),
    ]), list(numeric if numeric is not None else MODEL_NUMERIC_FEATURES))]
    if categorical:
        transformers.append(("cat", Pipeline([
            ("i", SimpleImputer(strategy="most_frequent")),
            ("o", OneHotEncoder(handle_unknown="ignore")),
        ]), list(categorical)))
    steps += [
        ("pre", ColumnTransformer(transformers)),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE, learning_rate=0.03, max_depth=2,
            min_samples_leaf=10, n_estimators=100)),
    ]
    return Pipeline(steps)

customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
events = pd.read_csv("../data/website_events.csv")
print("loaded")

from src.generate_data import CAMPAIGN_START_DAY
from src.scoring import load_model
from src.uplift import fit_t_learner, predict_uplift, qini_curve, qini_score, uplift_by_decile

campaign = pd.read_csv("../data/campaign.csv")
_, _, _, hidden_campaign = generate()
hidden, _, _, _ = generate()


loaded


## 2. The experiment

In [2]:

campaign_date = START_DATE + pd.Timedelta(days=CAMPAIGN_START_DAY - 1)
features = build_features(customers, orders, events, prediction_date=campaign_date)

panel = (features.merge(campaign, on="customer_id")
                 .merge(hidden_campaign[["customer_id", "true_uplift"]], on="customer_id"))

X = panel[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = panel["ordered_in_campaign"].to_numpy()
treated = panel["treated"].to_numpy()

print(f"campaign launched : {campaign_date.date()}")
print(f"randomised        : {treated.sum():,} treated / {(1-treated).sum():,} control")
print(f"response rate     : treated {y[treated==1].mean():.4f}   control {y[treated==0].mean():.4f}")
print(f"average treatment effect: {y[treated==1].mean() - y[treated==0].mean():+.4f}")
print()
print(f"true uplift is negative for {(panel['true_uplift'] < 0).mean():.1%} of customers")
print("(sleeping dogs -- contacting them makes things worse)")


campaign launched : 2026-03-31
randomised        : 2,503 treated / 2,497 control
response rate     : treated 0.4083   control 0.3628
average treatment effect: +0.0455

true uplift is negative for 14.9% of customers
(sleeping dogs -- contacting them makes things worse)



## 3. The T-learner

One outcome model per arm; predicted uplift is the difference. Each model sees
only its own arm, so neither can confuse "was treated" with "was going to
respond anyway".


In [3]:

def base_estimator():
    return Pipeline([
        ("rates", FunctionTransformer(add_rate_features)),
        ("pre", ColumnTransformer([
            ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                              ("s", StandardScaler())]), MODEL_NUMERIC_FEATURES),
            ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                              ("o", OneHotEncoder(handle_unknown="ignore"))]),
             CATEGORICAL_FEATURES),
        ])),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE, learning_rate=0.05, max_depth=3,
            min_samples_leaf=20, n_estimators=200)),
    ])

i_tr, i_te = train_test_split(
    np.arange(len(panel)), test_size=0.35,
    stratify=pd.Series(treated).astype(str) + pd.Series(y).astype(str),
    random_state=RANDOM_STATE)

models = fit_t_learner(base_estimator(), X.iloc[i_tr], treated[i_tr], y[i_tr])

X_te, y_te, t_te = X.iloc[i_te], y[i_te], treated[i_te]
uplift_pred = predict_uplift(models, X_te)

churn_risk = load_model().predict_proba(X_te)[:, 1]
random_score = np.random.default_rng(RANDOM_STATE).random(len(X_te))
true_uplift = panel["true_uplift"].to_numpy()[i_te]

print("fitted")


fitted


## 4. Four targeting strategies compared

In [4]:

strategies = {
    "uplift model (T-learner)": uplift_pred,
    "churn probability": churn_risk,
    "random": random_score,
    "ORACLE true uplift": true_uplift,
}

pd.DataFrame([
    {"strategy": name, "qini_score": round(qini_score(y_te, t_te, s), 2)}
    for name, s in strategies.items()
]).set_index("strategy")


,qini_score
strategy,
uplift model (T-learner),12.73
churn probability,1.44
random,1.98
ORACLE true uplift,14.62



The Qini score is the area between a strategy's curve and random targeting: how
many extra responders you gain by contacting people in this order rather than
arbitrarily.

**Targeting by churn probability scores no better than random.** That is the
headline. The model this project spent most of its effort on is, as a targeting
rule, worthless -- while the uplift model captures most of what the oracle
could.

## 5. Why: the two scores rank different people


In [5]:

for name in ("uplift model (T-learner)", "churn probability"):
    print(f"\n{name}")
    print(uplift_by_decile(y_te, t_te, strategies[name], bins=5)
          .round(3).to_string(index=False))



uplift model (T-learner)
 decile   n  treated_rate  control_rate  observed_uplift
      1 350         0.547         0.421            0.125
      2 350         0.382         0.285            0.097
      3 350         0.349         0.305            0.045
      4 350         0.376         0.400           -0.024
      5 350         0.394         0.394           -0.000

churn probability
 decile   n  treated_rate  control_rate  observed_uplift
      1 350         0.170         0.161            0.010
      2 350         0.330         0.238            0.092
      3 350         0.418         0.318            0.100
      4 350         0.457         0.475           -0.018
      5 350         0.685         0.604            0.080


In [6]:

pd.DataFrame([{
    "pair": "predicted uplift vs churn risk",
    "spearman": round(pd.Series(uplift_pred).corr(pd.Series(churn_risk), method="spearman"), 3),
}, {
    "pair": "predicted uplift vs TRUE uplift",
    "spearman": round(pd.Series(uplift_pred).corr(pd.Series(true_uplift), method="spearman"), 3),
}, {
    "pair": "churn risk vs TRUE uplift",
    "spearman": round(pd.Series(churn_risk).corr(pd.Series(true_uplift), method="spearman"), 3),
}]).set_index("pair")


,spearman
pair,
predicted uplift vs churn risk,-0.065
predicted uplift vs TRUE uplift,0.282
churn risk vs TRUE uplift,-0.202



The uplift model's quintiles fall cleanly from top to bottom and go negative --
it has found the sleeping dogs. The churn model's quintiles are not ordered at
all; its highest-risk customers are not its most persuadable ones.

The correlation between the two scores is near zero, which is by construction:
in this dataset persuadability is driven by how much a customer browses
relative to how much they buy, while churn risk is driven by how often they
order. Interested-but-hesitant customers respond to a nudge. Customers who
simply do not buy much are not persuadable, they are just quiet.

## 6. What this means

**The two questions are different, and only one of them is worth money.**
"Who will leave" is answerable from observational data and feels like the
natural modelling problem. "Whose mind can we change" requires an experiment
and is the question the budget actually turns on.

Three practical consequences:

1. **Run the experiment.** Uplift cannot be estimated from observational data
   at all. If no one was ever randomised, no amount of modelling recovers this.
2. **A churn model is still useful** -- for forecasting revenue at risk, for
   sizing the problem, for triage. Just not for deciding who to contact.
3. **Measure campaigns against a holdout, not against last quarter.** A
   campaign targeting sure things will show excellent retention among those
   contacted and will have achieved nothing.

The honest limitation: uplift is a difference of two noisy quantities, so it is
much harder to estimate than the outcome itself. The model here reaches a
Spearman correlation of only about 0.28 with true uplift, and still beats
random targeting by a wide margin -- weak signal on the right question beats
strong signal on the wrong one.
